In [1]:
import torch
import transformers
import datasets
import tokenizers
import trl
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

PyTorch device: cuda


# Stage 1

In [2]:
def generate_exprs():
    for a in range(2, 20 + 1):
        for b in range(2, 20 + 1):
            for c in range(2, 20 + 1):
                yield {
                    "expr": f"({a} + {b}) - {c}",
                    "result": (a + b) - c,
                    "reasoning": f"{a} + {b} = {a + b}. {a + b} - {c} = {(a + b) - c}."
                }
                if a >= b:
                    yield {
                        "expr": f"({a} - {b}) + {c}",
                        "result": (a - b) + c,
                        "reasoning": f"{a} - {b} = {a - b}. {a - b} + {c} = {(a - b) + c}."
                    }
                yield {
                    "expr": f"{a} + ({b} * {c})",
                    "result": a + (b * c),
                    "reasoning": f"{b} * {c} = {b * c}. {a} + {b * c} = {a + (b * c)}."
                }
                yield {
                    "expr": f"({a} * {b}) - c",
                    "result": (a * b) - c,
                    "reasoning": f"{a} * {b} = {a * b}. {a * b} - {c} = {(a * b) - c}."
                }
def convert_to_instructions(row):
    row["prompt"] = f"""
    Instruction: Solve the arithmetic expression step by step. Use exactly two lines:
    Reasoning: ...
    Answer: <integer>
    Expression: {row["expr"]}
    Response:
    """
    
    row["response"] = f"""
    Reasoning: {row["reasoning"]}
    Answer: {row["result"]}
    """
    return row

# hf's train_test_split cannot be used to create multiple split
# So create them manually
dataset = datasets.Dataset.from_generator(generate_exprs).map(convert_to_instructions).shuffle(seed=42)
dataset = datasets.DatasetDict({ "all": dataset })
SPLITS = [
    ("train_sft", 800),
    ("val_sft", 200),
    ("train_ppo", 100),
    ("test", 100)
]
skip_count = 0
for name, amount in SPLITS:
    dataset[name] = dataset["all"].skip(skip_count).take(amount)
    skip_count += amount
del SPLITS, skip_count, name, amount
del dataset["all"]
assert sum(map(len, dataset.values())) == 1200
print(dataset)

DatasetDict({
    train_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 800
    })
    val_sft: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 200
    })
    train_ppo: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 100
    })
    test: Dataset({
        features: ['expr', 'result', 'reasoning', 'prompt', 'response'],
        num_rows: 100
    })
})


# Stage 2

In [ ]:
base_name = "distilgpt2"
tok = AutoTokenizer.from_pretrained(base_name, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(base_name).to(device)
base_model.config.pad_token_id = tok.pad_token_id
